# Importing the Libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

### Finding the class distribution

In [ ]:
#To find the percentage of images in each folder

import os

data_path = 'drive/MyDrive/Combined New Dataset'
classes = [d for d in os.listdir(data_path) if os.path.isdir(os.path.join(data_path, d))]

print("True Class Distribution (Recursive):")
total_images = 0
counts = {}

for cls in classes:
    count = sum([len(files) for r, d, files in os.walk(os.path.join(data_path, cls))])
    counts[cls] = count
    total_images += count

for cls, count in counts.items():
    percentage = (count / total_images) * 100 if total_images > 0 else 0
    print(f"{cls}: {count} images ({percentage:.2f}%)")

# Data Preprocessing

Preparing the training data

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
train_datagen = ImageDataGenerator(rescale=1./255,shear_range=0.2,zoom_range=0.2,horizontal_flip=True,rotation_range=20, width_shift_range=0.1, height_shift_range=0.1, fill_mode='nearest')
training_set = train_datagen.flow_from_directory('drive/MyDrive/Combined New Dataset/train', target_size=(224,224), batch_size=32, class_mode='categorical', shuffle=True)

Preparing the test data

In [ ]:
test_datagen = ImageDataGenerator(rescale=1./255)
test_set = test_datagen.flow_from_directory('drive/MyDrive/Combined New Dataset/test', target_size=(224,224), batch_size=32, class_mode='categorical', shuffle=False)

### Load MobileNetV2 Base and add the custom head

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense, Dropout, BatchNormalization


NUM_CLASSES = training_set.num_classes

base_model = MobileNetV2( input_shape=(224,224) + (3,), include_top=False, weights='imagenet')
base_model.trainable = False

inputs = layers.Input(shape=(224,224) + (3,))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
x = BatchNormalization()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

tl_model = models.Model(inputs, outputs)

In [ ]:
print(training_set.class_indices)

COMPILING THE MobileNetV2 MODEL

In [ ]:
from tensorflow.keras.optimizers import Adam

tl_model.compile(optimizer="Adam", loss='categorical_crossentropy', metrics=['accuracy'])

Training the MobileNetV2 model

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping( monitor='val_loss', patience=3, restore_best_weights=True)
history_tl = tl_model.fit( training_set, validation_data=test_set, epochs=15, callbacks=[early_stopping])

Accuracy Checking and Plots

In [ ]:
loss, accuracy = tl_model.evaluate(test_set)
print(f"Test loss: {loss}")
print(f"Test accuracy: {accuracy}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

test_set.reset()
test_set.shuffle = False

predictions = tl_model.predict(test_set)
y_pred = np.argmax(predictions, axis=1)

y_true = test_set.classes
class_names = list(test_set.class_indices.keys())

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - Blood Classification')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()

# 6. Detailed Performance Report
print(classification_report(y_true, y_pred, target_names=class_names))

## Single Prediction

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image

image_path = '10.jpg'
img_width, img_height = 224, 224

test_image = image.load_img(image_path, target_size=(img_width, img_height))
test_image = image.img_to_array(test_image)

test_image = test_image / 255.0

test_image = np.expand_dims(test_image, axis=0)

result = tl_model.predict(test_image)

class_indices = training_set.class_indices
idx_to_class = {v: k for k, v in class_indices.items()}

predicted_index = np.argmax(result[0])
prediction = idx_to_class[predicted_index]

print(f"Predicted Type: {prediction}")
print(f"Confidence Scores: {result[0]}")

In [ ]:
tl_model.save('type_check_TL.h5')